# 💺 BUTACA EMOCIONAL · Versión 2 — Entrenamiento y gráficos reales

## ¿Qué vamos a construir?

En el **día 1** exploramos las reseñas. En la **versión 1** entrenamos un clasificador.

Ahora construiremos una **versión 2 independiente** que permitirá a Streamlit:

1. Clasificar reseñas como **positivas** o **negativas**.
2. Mostrar un gráfico con las **palabras que más influyen** en cada sentimiento.
3. Mostrar la **matriz de confusión** y el porcentaje de aciertos sobre reseñas de prueba.

Al terminar tendremos **dos archivos**: `mood_insights_modelo_v2.joblib` (modelo) y `metricas_v2.json` (resultados para los gráficos).

**Importante:** los gráficos se generarán con resultados reales, no con cifras inventadas. Conservaremos el modelo de la versión 1 sin modificarlo.

## 1. Instalar las herramientas

En Google Colab, una **biblioteca** es un conjunto de herramientas que podemos reutilizar. Primero las instalamos y después las importamos.

| Elemento | Qué hace | Por qué lo necesitamos |
|---|---|---|
| `%pip` | Ejecuta el instalador de Python desde el notebook. | Instalar paquetes en el entorno actual. |
| `install` | Pide instalar los paquetes indicados. | Disponer de sus funciones. |
| `-q` | *Quiet*: muestra menos mensajes de instalación. | Evitar que la pantalla se llene de texto. |
| `pandas` | Trabaja con tablas (`DataFrame`). | Leer y limpiar reseñas. |
| `scikit-learn` | Ofrece modelos y métricas de aprendizaje automático. | Entrenar y evaluar. |
| `joblib` | Guarda objetos de Python en un archivo. | Conservar el modelo entrenado. |
| `kagglehub` | Descarga datasets alojados en Kaggle. | Obtener las reseñas de cine. |

**Qué esperamos ver:** la celda terminará sin errores. No genera todavía un modelo ni una tabla.


In [32]:
# %pip ejecuta pip dentro de Google Colab o Jupyter.
# -q reduce los mensajes mostrados mientras se instalan las bibliotecas.
# Solo necesitamos instalar; todavía NO entrenamos el modelo.
%pip -q install pandas scikit-learn joblib kagglehub


## 2. Importar herramientas y conocer el entorno

Instalar una biblioteca **no es lo mismo que importarla**: `import` permite utilizarla en el código. `from ... import ...` trae una función o clase concreta.

| Importación | Para qué sirve |
|---|---|
| `json` | Convertir datos Python al formato JSON que leerá Streamlit. |
| `sys` | Consultar la versión de Python. |
| `Path` de `pathlib` | Manejar rutas y carpetas sin escribir separadores manualmente. |
| `joblib` | Guardar la Pipeline entrenada. |
| `pandas as pd` | Gestionar las reseñas como una tabla. `pd` es un alias. |
| `sklearn` | Consultar la versión instalada de Scikit-learn. |
| `display` | Presentar tablas con formato en el notebook. |
| `train_test_split` | Dividir datos para aprendizaje y examen. |
| `CountVectorizer` | Transformar textos en números mediante conteos. |
| `LogisticRegression` | Clasificar las reseñas según esos números. |
| `Pipeline` | Encadenar la transformación de texto y la clasificación. |
| `accuracy_score` | Calcular la proporción global de aciertos. |
| `classification_report` | Calcular precisión y recall por clase. |
| `confusion_matrix` | Contar aciertos y errores de cada clase. |

`sys.version.split()[0]` devuelve la primera parte del texto de versión; `sklearn.__version__` devuelve la versión de la biblioteca. Guardarlas ayuda a investigar posibles incompatibilidades al abrir el modelo en Streamlit.


In [33]:
# Módulos de Python incluidos en el lenguaje.
import json                   # Permite escribir un diccionario en formato .json.
import sys                    # Permite consultar datos sobre el intérprete Python.
from pathlib import Path      # Manejo sencillo de rutas y carpetas.

# Bibliotecas que hemos instalado en el paso anterior.
import joblib                 # Guarda el modelo entrenado como archivo .joblib.
import pandas as pd           # Tablas de datos. El alias "pd" es el habitual.
import sklearn                # Consultamos la versión de Scikit-learn.
from IPython.display import display  # Muestra tablas de forma legible en Colab.

# División de los datos, conversión de textos y aprendizaje del modelo.
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# Funciones para evaluar los resultados.
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# split() divide una cadena en partes; [0] selecciona la primera.
print("Python:", sys.version.split()[0])
# __version__ guarda la versión instalada de Scikit-learn.
print("Scikit-learn:", sklearn.__version__)


Python: 3.13.18
Scikit-learn: 1.6.1


## 3. Descargar y cargar las reseñas de Kaggle

Utilizamos el dataset **`luisdiegofv97/imdb-dataset-of-50k-movie-reviews-spanish`**. La descarga devuelve una carpeta; debemos localizar el archivo CSV que contiene la información.

**Métodos y funciones que aparecen en esta celda:**

- `kagglehub.dataset_download(...)`: descarga o recupera de la caché el dataset de Kaggle; devuelve su ruta.
- `Path(...)`: convierte esa ruta en un objeto que permite trabajar con carpetas.
- `.rglob("*.csv")`: busca archivos terminados en `.csv`, incluso dentro de subcarpetas.
- `list(...)`: convierte el resultado de la búsqueda en una lista de rutas.
- `if not archivos`: comprueba si la lista está vacía.
- `raise FileNotFoundError(...)`: detiene el proceso con un error explicativo si no hay CSV.
- `next(..., valor_alternativo)`: elige el primer elemento que cumple una condición; si ninguno cumple, usa la alternativa indicada.
- `.name`: obtiene el nombre del archivo, sin toda la ruta.
- `.upper()`: convierte texto a mayúsculas para comparar sin distinguir `imdb` de `IMDB`.
- `pd.read_csv(...)`: lee el CSV y lo convierte en una tabla de pandas.
- `[["review_es", "sentimiento"]]`: selecciona solamente esas dos columnas.
- `.rename(columns={...})`: cambia nombres de columnas sin modificar las reseñas.
- `.issubset(...)`: comprueba si todas las columnas necesarias existen.
- `.shape`: devuelve `(filas, columnas)`.
- `.head(3)`: muestra las tres primeras filas, para comprobar que hemos leído lo esperado.

**Ejemplo:** si `df.shape` devuelve `(50000, 2)`, significa que hay 50.000 filas y 2 columnas. La cifra exacta depende del CSV descargado.

**Atención:** este código espera que el CSV seleccionado contenga `review_es` y `sentimiento`. Si el archivo cambia de estructura, habrá que adaptar la lectura; el segundo `rename()` no puede reparar columnas ausentes después de un error en `read_csv(...)[[...]]`.


In [34]:
# Importamos la herramienta para descargar el dataset desde Kaggle.
import kagglehub

# dataset_download() obtiene una carpeta con los archivos del dataset.
# Path() nos permite navegar por ella con métodos de pathlib.
carpeta = Path(kagglehub.dataset_download("luisdiegofv97/imdb-dataset-of-50k-movie-reviews-spanish"))

# rglob("*.csv") encuentra cualquier CSV en esta carpeta y sus subcarpetas.
# list() guarda todas las rutas encontradas en una lista.
archivos = list(carpeta.rglob("*.csv"))

# Si no hay CSV, detenemos el notebook en lugar de seguir con datos inexistentes.
if not archivos:
    raise FileNotFoundError("No se encontró un CSV. Sube mood_insights_limpio.csv.")

# next() intenta elegir un nombre que contenga IMDB; si no existe, usa el primer CSV.
# p.name es el nombre del archivo y upper() lo convierte a mayúsculas.
archivo = next((p for p in archivos if "IMDB" in p.name.upper()), archivos[0])

# read_csv() lee la tabla; [[...]] conserva las dos columnas que vamos a usar.
# rename() deja el nombre "reseña" en español.
df = pd.read_csv(archivo)[["review_es", "sentimiento"]].rename(columns={"review_es": "reseña"})
print("Usamos el CSV original:", archivo.name)

# Este rename adicional acepta nombres alternativos SI aparecen en la tabla.
df = df.rename(columns={"reseñas": "reseña", "sentimientos": "sentimiento", "review_es": "reseña"})

# set() crea un conjunto con los nombres de las columnas obligatorias.
# issubset() comprueba que todas estén presentes en df.columns.
columnas = {"reseña", "sentimiento"}
if not columnas.issubset(df.columns):
    raise ValueError(f"El CSV debe tener las columnas {columnas}. Encontradas: {list(df.columns)}")

# shape informa del número de filas y columnas.
print("Datos cargados:", df.shape)
# head(3) selecciona las primeras 3 reseñas y display() las muestra bien formateadas.
display(df[["reseña", "sentimiento"]].head(3))


Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews-spanish' dataset.
Usamos el CSV original: IMDB Dataset SPANISH.csv
Datos cargados: (50000, 2)


,reseña,sentimiento
0,Uno de los otros críticos ha mencionado que de...,positivo
1,Una pequeña pequeña producción.La técnica de f...,positivo
2,Pensé que esta era una manera maravillosa de p...,positivo


### ✅ Resultado real: carga del conjunto de datos

El método `dataset_download()` obtiene el conjunto de reseñas de Kaggle. A continuación, `pd.read_csv()` abre el archivo CSV y construye una tabla (*DataFrame*).

**Resultado de la ejecución:** se cargaron **50.000 reseñas y 2 columnas**: `reseña` (el texto de la opinión) y `sentimiento` (la etiqueta positivo o negativo). `df.shape` devuelve `(50000, 2)`: el primer número son las filas y el segundo las columnas. `head(3)` muestra las primeras tres filas para comprobar que el archivo se ha leído correctamente.

## 4. Limpieza mínima y comprobación

Antes de entrenar debemos evitar que una reseña repetida aparezca en entrenamiento y prueba.

1. Quitamos filas sin reseña o sentimiento.
2. Conservamos las etiquetas **positivo** y **negativo**.
3. Quitamos espacios al comienzo y al final.
4. Eliminamos duplicados y textos con etiquetas contradictorias.

**No quitamos stopwords ni lematizamos**: algunas palabras como «no», «muy» y «pero» pueden influir en el sentimiento.
### Diccionario de métodos del siguiente código

| Método | Explicación sencilla | Mini ejemplo |
|---|---|---|
| `dropna()` | Elimina filas con valores vacíos (`NaN`) en las columnas seleccionadas. | Una fila sin reseña se descarta. |
| `copy()` | Crea una copia independiente de la tabla seleccionada. | Podemos modificar la copia. |
| `astype(str)` | Convierte los valores de una columna a texto. | `12` pasa a `"12"`. |
| `str.strip()` | Quita espacios al principio y al final. | `" hola "` pasa a `"hola"`. |
| `str.lower()` | Convierte el texto a minúsculas. | `"POSITIVO"` pasa a `"positivo"`. |
| `isin([...])` | Comprueba si cada valor pertenece a la lista indicada. | Acepta positivo o negativo. |
| `drop_duplicates(subset=[...])` | Deja una sola fila de cada combinación repetida de columnas. | Dos filas con igual reseña e igual etiqueta → una. |
| `groupby("reseña")` | Agrupa las filas que tienen exactamente el mismo texto. | Todas las filas de `"Me gustó"` juntas. |
| `nunique()` | Cuenta cuántos valores **distintos** hay en cada grupo. | Dos etiquetas distintas → resultado 2. |
| `reset_index(drop=True)` | Vuelve a numerar las filas desde cero sin guardar el índice antiguo. | Índices 0, 3, 8 → 0, 1, 2. |
| `value_counts()` | Cuenta las veces que aparece cada categoría. | Cuántas reseñas son positivas. |
| `to_frame()` | Convierte una serie de pandas en una tabla de una columna. | Conteos presentados como tabla. |

**Por qué quitamos textos contradictorios:** si una misma reseña está marcada como positiva y negativa, el modelo recibiría dos respuestas incompatibles para un texto idéntico. No eliminamos palabras como «no» porque pueden cambiar el sentido.


In [35]:
# Seleccionamos las columnas necesarias y descartamos filas que tengan datos ausentes.
# copy() crea una tabla nueva para trabajar sobre ella sin ambigüedades.
df = df[["reseña", "sentimiento"]].dropna().copy()

# astype(str) convierte cada valor a texto; str.strip() elimina espacios externos.
df["reseña"] = df["reseña"].astype(str).str.strip()
# Las etiquetas deben escribirse igual: quitamos espacios y pasamos a minúsculas.
df["sentimiento"] = df["sentimiento"].astype(str).str.strip().str.lower()

# & exige que se cumplan las dos condiciones: reseña no vacía y etiqueta válida.
# isin() es la comprobación de pertenencia a ["positivo", "negativo"].
df = df[(df["reseña"] != "") & (df["sentimiento"].isin(["positivo", "negativo"]))]

# Elimina duplicados exactos de (reseña, sentimiento), conservando el primero.
df = df.drop_duplicates(subset=["reseña", "sentimiento"])

# groupby() reúne reseñas idénticas; nunique() cuenta las etiquetas diferentes.
# Si una misma reseña tiene 2 etiquetas distintas, hay una contradicción.
conflictos = df.groupby("reseña")["sentimiento"].nunique()

# conflictos[conflictos > 1].index contiene los textos contradictorios.
# isin() los localiza; ~ niega la condición y conserva SOLO los demás.
# reset_index(drop=True) vuelve a numerar las filas desde 0.
df = df[~df["reseña"].isin(conflictos[conflictos > 1].index)].reset_index(drop=True)

print("Total de reseñas listas:", len(df))  # len() cuenta filas.
print("Número de textos contradictorios eliminados:", (conflictos > 1).sum())
# value_counts() cuenta positivos y negativos; rename() titula la serie.
# to_frame() la convierte en tabla y display() la muestra.
display(df["sentimiento"].value_counts().rename("Número de reseñas").to_frame())


Total de reseñas listas: 49599
Número de textos contradictorios eliminados: 0


,Número de reseñas
sentimiento,
positivo,24886
negativo,24713


### ✅ Resultado real: limpieza de los datos

Tras eliminar filas no válidas y aplicar la limpieza prevista, quedan **49.599 reseñas**. Se han descartado **401 registros** respecto a los 50.000 cargados. El resultado informa también de **0 textos contradictorios eliminados** (textos idénticos asociados a etiquetas diferentes), lo que no significa que no se hayan eliminado registros por otros motivos.

| Sentimiento | Reseñas limpias |
|---|---:|
| Positivo | 24.886 |
| Negativo | 24.713 |

Las dos clases están bastante equilibradas; ninguna domina el conjunto de datos.

**Métodos empleados:** `dropna()` elimina filas con valores vacíos; `astype(str)` convierte valores a texto; `str.strip()` elimina espacios de los extremos; `value_counts()` cuenta las reseñas de cada clase. Otros filtros del código pueden descartar registros adicionales.

## 5. Separar entrenamiento y prueba

Piensa en un examen: **no debemos enseñar las respuestas del examen al alumnado antes de evaluar**.

- **80 %** de las reseñas: el modelo aprende con ellas.
- **20 %**: las guardamos para comprobar si clasifica opiniones desconocidas.

`stratify=y` mantiene una proporción parecida de opiniones positivas y negativas en ambos grupos. `random_state=42` permite reproducir la misma división.
### Cómo leer `train_test_split()`

La función devuelve **cuatro grupos**: `X_train` (textos para aprender), `X_test` (textos para el examen), `y_train` (etiquetas correctas del aprendizaje) y `y_test` (etiquetas correctas del examen).

- `X` significa **entradas**: los textos de las reseñas.
- `y` significa **objetivo** o respuesta correcta: positivo/negativo.
- `test_size=0.20`: reserva el 20 % para evaluar.
- `random_state=42`: fija la semilla aleatoria. Si repetimos la ejecución con los mismos datos, obtendremos la misma separación.
- `stratify=y`: busca conservar proporciones similares de ambas etiquetas en los dos grupos.
- `len(...)`: cuenta cuántas reseñas hay en cada grupo.

**Ejemplo inventado solo para comprender:** con 100 reseñas, aproximadamente 80 entrenan y 20 se reservan para el examen.


In [36]:
# X son los datos de entrada: las frases que el modelo va a leer.
X = df["reseña"]
# y es la etiqueta verdadera de cada frase: positivo o negativo.
y = df["sentimiento"]

# train_test_split() separa X e y SIN romper la correspondencia entre cada reseña y su etiqueta.
# test_size=0.20 -> 20% para prueba y 80% para entrenamiento.
# random_state=42 -> misma selección aleatoria si repetimos con los mismos datos.
# stratify=y -> proporción de etiquetas semejante en ambos grupos.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# len() nos dice el tamaño real de cada conjunto.
print("Reseñas para aprender:", len(X_train))
print("Reseñas reservadas para el examen:", len(X_test))


Reseñas para aprender: 39679
Reseñas reservadas para el examen: 9920


### ✅ Resultado real: separar datos para aprender y evaluar

`train_test_split()` divide los datos en **39.679 reseñas de entrenamiento (80 %)** y **9.920 reseñas de prueba (20 %)**. El modelo aprende únicamente con las primeras y se examina con las segundas.

- `test_size=0.20`: reserva el 20 % para la prueba.
- `random_state=42`: fija la selección aleatoria para poder reproducir la misma partición.
- `stratify=y`: mantiene aproximadamente la proporción de reseñas positivas y negativas en ambos grupos.

**Por qué importa:** evaluar con las mismas reseñas empleadas para entrenar daría una impresión poco fiable de su capacidad para analizar opiniones nuevas.

## 6. Crear el modelo de la versión 2

Una **Pipeline** es una cadena de dos pasos que siempre se ejecutan en orden:

**Paso A — CountVectorizer.** Lee los textos, construye el vocabulario y cuenta términos. En esta segunda versión incluimos también algunas **parejas de palabras** (`ngram_range=(1,2)`). Así puede reconocer expresiones como «no gusta», en lugar de mirar siempre palabras aisladas.

**Paso B — LogisticRegression.** Aprende qué patrones suelen relacionarse con cada sentimiento.

- `min_df=2` evita tokens que aparecen en una sola reseña de entrenamiento.
- `max_features=40000` limita el tamaño del vocabulario para simplificar el modelo y su archivo.
- `max_iter=1000` es un **máximo de iteraciones del optimizador**, no un número de epochs.

No garantizamos que esta versión sea más precisa: lo comprobaremos con el examen.
### Qué hace cada método o parámetro

| Elemento | Qué hace |
|---|---|
| `Pipeline([...])` | Une varios pasos en un único objeto. Al llamar a `fit()` o `predict()`, ejecuta los pasos pertinentes en orden. |
| `CountVectorizer(...)` | Aprende un vocabulario de los textos de entrenamiento y convierte cada reseña en conteos numéricos. |
| `ngram_range=(1, 2)` | Incluye palabras aisladas (*unigramas*) y parejas consecutivas (*bigramas*). |
| `min_df=2` | Solo incorpora términos presentes en al menos dos documentos de entrenamiento. |
| `max_features=40000` | Limita el vocabulario a un máximo de 40.000 características. |
| `LogisticRegression(...)` | Aprende los coeficientes para decidir entre dos clases; aunque se llame regresión, aquí se usa para **clasificación**. |
| `max_iter=1000` | Permite hasta 1.000 iteraciones al optimizador si las necesita. |
| `fit(X_train, y_train)` | Ajusta el vocabulario y aprende los coeficientes utilizando **solo** los datos de entrenamiento. |
| `named_steps["contador"]` | Recupera la etapa del contador por el nombre que le hemos dado. |
| `get_feature_names_out()` | Devuelve los términos del vocabulario aprendido. |

**Un ejemplo de transformación:** «película excelente» puede convertirse en conteos para los términos «película», «excelente» y «película excelente», siempre que esas características pertenezcan al vocabulario aprendido. El modelo trabaja con números, no entiende las palabras como una persona.


In [37]:
# Pipeline() encadena dos herramientas y les asigna nombres propios.
modelo_v2 = Pipeline([
    # Primera etapa, "contador": convertir textos en números.
    ("contador", CountVectorizer(
        ngram_range=(1, 2),   # Usa palabras sueltas (1) y parejas de palabras (2).
        min_df=2,             # El término debe aparecer en al menos dos reseñas de entrenamiento.
        max_features=40000    # Como máximo se guardan 40.000 términos.
    )),
    # Segunda etapa, "clasificador": aprender cómo se relacionan los números con la etiqueta.
    # max_iter limita los intentos del optimizador; NO significa 1.000 reseñas.
    ("clasificador", LogisticRegression(max_iter=1000, random_state=42))
])

# fit() aprende con X_train (textos) e y_train (respuestas correctas).
# La Pipeline primero prepara los conteos y luego ajusta LogisticRegression.
# No le enseñamos X_test, que está reservado para el examen.
modelo_v2.fit(X_train, y_train)

print("✅ Versión 2 entrenada")
# named_steps elige la etapa "contador" de la Pipeline.
# get_feature_names_out() devuelve todos los términos aprendidos; len() los cuenta.
print("Términos del vocabulario:", len(modelo_v2.named_steps["contador"].get_feature_names_out()))


✅ Versión 2 entrenada
Términos del vocabulario: 40000


### ✅ Resultado real: entrenamiento completado

La salida confirma que el entrenamiento terminó y que el vocabulario contiene **40.000 términos**. Un término puede ser una palabra o una pareja de palabras, ya que el vectorizador admite ambas.

`Pipeline()` une el vectorizador y el clasificador; `fit(X_train, y_train)` aprende las relaciones entre las reseñas y sus etiquetas. `named_steps["contador"]` permite acceder al paso que transforma el texto y `get_feature_names_out()` devuelve su vocabulario.

**Importante:** 40.000 términos no significa que el modelo haya entendido 40.000 conceptos como lo haría una persona: son características matemáticas obtenidas del texto.

## 7. Evaluar sin utilizar los datos para entrenar

`predict()` devuelve las categorías que estima el modelo para las reseñas del examen. Después las comparamos con sus etiquetas reales.

**Exactitud (accuracy)**: proporción de respuestas correctas. No es lo mismo que probabilidad de acertar en todos los textos futuros.

**Matriz de confusión**: tabla que compara las respuestas verdaderas con las predicciones; veremos qué clases se confunden más.
### Cómo interpretaremos las métricas

| Método | Entrada | Salida |
|---|---|---|
| `predict(X_test)` | Textos desconocidos para el modelo. | Etiqueta estimada por reseña. |
| `accuracy_score(y_test, predicciones)` | Etiquetas reales y estimadas. | Número entre 0 y 1: proporción de aciertos. |
| `confusion_matrix(..., labels=etiquetas)` | Etiquetas reales, estimadas y orden de clases. | Tabla 2 × 2 de aciertos y errores. |
| `classification_report(..., output_dict=True)` | Etiquetas reales y estimadas. | Diccionario con precisión, recall y otras medidas. |
| `zero_division=0` | Control de casos donde una métrica no se puede dividir. | Devuelve 0 en vez de avisar por división entre cero. |
| `pd.DataFrame(...)` | Datos, nombres de filas y columnas. | Tabla legible para mostrar la matriz. |
| `f"{exactitud:.2%}"` | Un número como `0.87`. | Texto `87.00%` (ejemplo, no resultado real). |

**Matriz de confusión:** la primera fila representa reseñas realmente negativas; la segunda, realmente positivas. La primera columna es lo que el modelo predijo como negativo y la segunda, como positivo. **Las casillas de la diagonal son aciertos**; las otras dos son errores.

**Precisión:** de lo que el modelo clasificó como una categoría, ¿qué proporción pertenecía realmente a ella? **Recall:** de las reseñas que realmente pertenecen a una categoría, ¿cuántas detectó?


In [38]:
# predict() devuelve la clase que el modelo estima para cada reseña NO vista en entrenamiento.
predicciones = modelo_v2.predict(X_test)

# accuracy_score() compara respuestas correctas (y_test) con predicciones.
# Por ejemplo, 0.90 significaría 90% de aciertos sobre ESTE conjunto de prueba.
exactitud = accuracy_score(y_test, predicciones)

# Fijamos el orden de las dos etiquetas en las tablas y el JSON.
etiquetas = ["negativo", "positivo"]

# confusion_matrix() cuenta los casos de cada combinación real/predicha.
# Filas: valores reales. Columnas: valores predichos.
matriz = confusion_matrix(y_test, predicciones, labels=etiquetas)

# classification_report() calcula, entre otras medidas, precisión y recall por clase.
# output_dict=True devuelve un diccionario, más fácil de reutilizar después.
# zero_division=0 evita un aviso si en algún caso el denominador es 0.
informe = classification_report(y_test, predicciones, labels=etiquetas, output_dict=True, zero_division=0)

# :.2% convierte la proporción a porcentaje con 2 decimales.
print(f"Exactitud sobre el examen: {exactitud:.2%}")
print("Matriz de confusión (filas: sentimiento real; columnas: predicción):")
# DataFrame() añade nombres a filas y columnas de la matriz para entenderla.
display(pd.DataFrame(matriz, index=["Real negativo", "Real positivo"], columns=["Predicho negativo", "Predicho positivo"]))


Exactitud sobre el examen: 87.64%
Matriz de confusión (filas: sentimiento real; columnas: predicción):


,Predicho negativo,Predicho positivo
Real negativo,4294,649
Real positivo,577,4400


### ✅ Resultado real: evaluación del modelo

La exactitud obtenida en las **9.920 reseñas que el modelo no utilizó para entrenar** es del **87,64 %**. Eso significa que acertó en **8.694 reseñas** y falló en **1.226**.

| Sentimiento real | Predijo negativo | Predijo positivo |
|---|---:|---:|
| Negativo | **4.294 aciertos** | 649 errores |
| Positivo | 577 errores | **4.400 aciertos** |

**Cómo leer la matriz de confusión:** las filas indican lo que realmente era la reseña y las columnas lo que predijo el modelo. Los dos números de la diagonal (**4.294** y **4.400**) son clasificaciones correctas. Los otros dos (**649** y **577**) son errores.

`predict()` genera predicciones; `accuracy_score()` calcula la fracción de predicciones correctas; `confusion_matrix()` muestra dónde acertó y dónde confundió una clase con la otra. La exactitud es una medida sobre este conjunto de prueba, no una garantía de acierto en todas las frases nuevas.

## 8. ¿Qué palabras ha aprendido que influyen en las decisiones?

El clasificador guarda un **coeficiente** por término:

- **Coeficiente positivo**: favorece la categoría positiva.
- **Coeficiente negativo**: favorece la categoría negativa.

Cuanto mayor es su magnitud, más empuja la decisión **en este modelo**, sin que eso implique que la palabra siempre tenga ese sentimiento. Estas palabras provienen del entrenamiento y son distintas de las palabras simplemente más frecuentes.
### Leer los términos y ordenar sus pesos

| Método o propiedad | Significado |
|---|---|
| `named_steps["contador"]` | Accede al vectorizador que aprendió las palabras. |
| `named_steps["clasificador"]` | Accede al algoritmo que aprendió los pesos. |
| `get_feature_names_out()` | Devuelve una lista de términos alineados con los coeficientes. |
| `coef_[0]` | Devuelve el vector de coeficientes para la clasificación binaria. |
| `classes_` | Muestra las etiquetas en el orden interno del clasificador. |
| `np.argsort(coeficientes)` | Devuelve **posiciones** que ordenarían los pesos de menor a mayor (no devuelve los pesos). |
| `orden[:12]` | Escoge las 12 posiciones de los pesos más negativos. |
| `orden[-12:][::-1]` | Toma las 12 posiciones de los más positivos y las invierte para ver primero el mayor. |
| `float(...)`, `str(...)` | Convierten los valores a números/textos sencillos que puedan guardarse en JSON. |
| `round(..., 4)` | Redondea un número a cuatro decimales. |
| `pd.DataFrame(lista)` | Convierte la lista de diccionarios en una tabla. |

**Ejemplo inventado:** para coeficientes `[0.6, -1.2, 0.1]`, `argsort()` devuelve posiciones `[1, 2, 0]`. La posición `1` contiene el peso más negativo. **No significa** que esa palabra sea siempre negativa: es una asociación estadística de este modelo.


In [39]:
# Accedemos a cada uno de los dos pasos mediante su nombre dentro de la Pipeline.
vectorizador = modelo_v2.named_steps["contador"]
clasificador = modelo_v2.named_steps["clasificador"]

# get_feature_names_out() devuelve los términos en el mismo orden que sus pesos.
terminos = vectorizador.get_feature_names_out()
# coef_[0] es la fila de coeficientes que utiliza esta clasificación binaria.
coeficientes = clasificador.coef_[0]

# classes_ informa de las etiquetas que el algoritmo conoce y su orden.
print("Clases del modelo:", list(clasificador.classes_))
# En este ejemplo, pesos positivos deben favorecer "positivo"; comprobamos el orden.
if list(clasificador.classes_) != ["negativo", "positivo"]:
    raise ValueError("Las clases no tienen el orden previsto; revisa la interpretación de coeficientes.")

# NumPy facilita ordenar y seleccionar índices de arrays numéricos.
import numpy as np
# argsort() devuelve las POSICIONES ordenadas del peso más negativo al más positivo.
orden = np.argsort(coeficientes)

# orden[-12:] toma los 12 índices más positivos; [::-1] los invierte.
# str() y float() convierten los valores a tipos básicos; round(..., 4) redondea.
positivas = [{"termino": str(terminos[i]), "peso": round(float(coeficientes[i]), 4)} for i in orden[-12:][::-1]]
# orden[:12] toma los 12 índices de peso más negativo.
negativas = [{"termino": str(terminos[i]), "peso": round(float(coeficientes[i]), 4)} for i in orden[:12]]

print("Términos que más favorecen una opinión positiva:")
display(pd.DataFrame(positivas))  # Lista de diccionarios -> tabla.
print("Términos que más favorecen una opinión negativa:")
display(pd.DataFrame(negativas))


Clases del modelo: ['negativo', 'positivo']
Términos que más favorecen una opinión positiva:


,termino,peso
0,refrescante,1.3725
1,perfecto,1.2730
2,puedo esperar,1.2704
3,excelente,1.2585
4,los mejores,1.2547
5,bien hecho,1.2175
6,perfecta,1.2172
7,un poco,1.1733
8,maravilloso,1.1427
9,hilarante,1.1371


Términos que más favorecen una opinión negativa:


,termino,peso
0,decepción,-1.9333
1,no vale,-1.8277
2,horrible,-1.7974
3,aburrida,-1.7865
4,decepcionante,-1.6975
5,olvidable,-1.6614
6,terrible,-1.6222
7,peor,-1.5700
8,peores,-1.5300
9,mst3k,-1.4734


### ✅ Resultado real: términos asociados a cada sentimiento

Los resultados muestran que **«refrescante» (+1,3725)**, **«perfecto» (+1,2730)** y **«excelente» (+1,2585)** son algunos de los términos con más peso hacia *positivo*. En cambio, **«decepción» (−1,9333)**, **«no vale» (−1,8277)** y **«horrible» (−1,7974)** tienen un peso destacado hacia *negativo*.

**Qué significan los pesos:** los valores positivos inclinan la decisión hacia la clase positiva; los negativos, hacia la negativa. No son porcentajes ni cantidad de veces que aparece una palabra. Expresiones como «un poco» pueden aparecer entre los pesos positivos porque así se asociaron en estos datos; **no significa que siempre expresen una emoción positiva**.

`named_steps` recupera las etapas del modelo; `coef_` contiene los pesos matemáticos aprendidos; `argsort()` ordena los índices según esos pesos; `DataFrame()` organiza los resultados en tablas legibles.

## 9. Guardar los DOS archivos que necesita la aplicación

**Archivo 1 — `.joblib`:** guarda la Pipeline completa (el contador y el clasificador). La aplicación podrá hacer predicciones.

**Archivo 2 — `.json`:** guarda la exactitud, la matriz de confusión y los términos importantes. La aplicación podrá mostrar gráficos sin volver a entrenar.

El JSON también registra las versiones de Python y Scikit-learn para facilitar la comprobación de compatibilidad.

**No guardamos las reseñas originales ni datos privados en estos archivos.**
### Qué hace cada método al guardar

- `joblib.dump(modelo_v2, archivo_modelo)` escribe el modelo **ya entrenado** en un archivo `.joblib` para reutilizarlo.
- `len(...)` cuenta elementos, como el número de reseñas utilizadas.
- `int(...)` y `float(...)` convierten valores a tipos numéricos básicos aptos para JSON.
- `matriz.tolist()` convierte una matriz de NumPy en listas de Python (JSON no guarda directamente arrays de NumPy).
- `informe[clase]["precision"]` y `informe[clase]["recall"]` extraen las métricas de cada etiqueta.
- `{clase: ... for clase in etiquetas}` es una **comprensión de diccionario**: crea una entrada por etiqueta.
- `with open(..., "w", encoding="utf-8") as f` abre un archivo para escribir y lo cierra automáticamente al terminar. La `w` significa *write* y sustituye el contenido anterior.
- `json.dump(metricas, f, ensure_ascii=False, indent=2)` escribe el diccionario en JSON: `ensure_ascii=False` conserva caracteres como `ñ`, e `indent=2` lo formatea con sangrías para leerlo mejor.

El archivo `.joblib` sirve para **predecir nuevas reseñas**. El `.json` sirve para **mostrar métricas y gráficos del entrenamiento**. Guardamos dos cosas distintas porque cumplen funciones distintas.


In [40]:
# 1) GUARDAR EL MODELO QUE YA ENTRENAMOS
archivo_modelo = "butaca_emocional_modelo_v2.joblib"
# dump() serializa (guarda) la Pipeline, incluyendo contador y clasificador.
joblib.dump(modelo_v2, archivo_modelo)

# 2) PREPARAR LAS MÉTRICAS PARA LA APLICACIÓN
# Este diccionario contiene números, listas y textos que después leerá Streamlit.
metricas = {
    "version_modelo": "2",                              # Identificador de esta versión.
    "python": sys.version.split()[0],                   # Versión del intérprete Python.
    "scikit_learn": sklearn.__version__,                 # Versión de Scikit-learn.
    "n_entrenamiento": int(len(X_train)),                # Total de reseñas para aprender.
    "n_prueba": int(len(X_test)),                         # Total de reseñas del examen.
    "tamano_vocabulario": int(len(terminos)),            # Cantidad de términos aprendidos.
    "exactitud": float(exactitud),                        # Proporción de predicciones acertadas.
    "etiquetas": etiquetas,                              # Orden: negativo, positivo.
    "matriz_confusion": matriz.tolist(),                 # Array de NumPy -> listas normales.
    # Comprensión de diccionario: una métrica por cada etiqueta.
    "precision_por_clase": {clase: float(informe[clase]["precision"]) for clase in etiquetas},
    "recall_por_clase": {clase: float(informe[clase]["recall"]) for clase in etiquetas},
    "terminos_positivos": positivas,                     # Palabras con mayores pesos positivos.
    "terminos_negativos": negativas                      # Palabras con mayores pesos negativos.
}

archivo_metricas = "metricas_v2.json"
# with se ocupa de cerrar el archivo automáticamente cuando terminamos.
# "w" significa escribir; encoding="utf-8" admite letras y acentos.
with open(archivo_metricas, "w", encoding="utf-8") as f:
    # dump() convierte el diccionario a JSON y lo escribe en el archivo f.
    # ensure_ascii=False conserva la ñ; indent=2 hace el JSON legible.
    json.dump(metricas, f, ensure_ascii=False, indent=2)

print("✅ Modelo guardado:", archivo_modelo)
print("✅ Métricas guardadas:", archivo_metricas)


✅ Modelo guardado: butaca_emocional_modelo_v2.joblib
✅ Métricas guardadas: metricas_v2.json


### ✅ Resultado real: archivos generados

La ejecución confirma la creación de **`butaca_emocional_modelo_v2.joblib`** y **`metricas_v2.json`**.

- **`.joblib`**: guarda la canalización entrenada que clasifica reseñas nuevas.
- **`.json`**: guarda las métricas y los términos para mostrar los resultados y gráficos en Streamlit.

`joblib.dump()` guarda el modelo. `json.dump()` escribe datos estructurados en el archivo JSON. Estos archivos se deben conservar junto a la aplicación, que debe apuntar a sus nombres exactos.

## 10. Descargar y colocar en la carpeta de Streamlit

Al ejecutar la siguiente celda se descargarán los dos archivos. Guárdalos en **`BUTACA_EMOCIONAL_VERSION_2/`**, junto a `app.py`:

```text
BUTACA_EMOCIONAL_VERSION_2/
├── app.py
├── requirements.txt
├── README.md
├── butaca_emocional_modelo_v2.joblib  ← lo generamos ahora
└── metricas_v2.json              ← lo generamos ahora
```

Nunca abras un archivo `.joblib` de una fuente desconocida: cargarlo ejecuta deserialización de Python.

**Compatibilidad:** si Colab instala una versión de Scikit-learn diferente a la que tienes en Python 3.14.3, cargar el archivo en Windows puede fallar o dar advertencias. La solución fiable es entrenar y ejecutar con la **misma versión compatible** de Scikit-learn. No garantizamos compatibilidad cruzada.
### Métodos de la descarga

- `from google.colab import files`: importa la herramienta de archivos propia de **Google Colab**.
- `files.download("nombre")`: solicita al navegador descargar un archivo que existe en la sesión de Colab.
- Ejecutamos `download()` dos veces porque la aplicación necesita **ambos archivos**.
- Este método es específico de Colab. Si ejecutas el notebook en Jupyter local, la descarga se gestiona de otra manera: los archivos ya se encuentran en la carpeta donde se ejecutó el código.

**Comprobación:** después de descargar, confirma que ambos nombres coinciden exactamente con los que busca `app.py`. No hace falta volver a entrenar para cada reseña nueva: la aplicación carga el `.joblib` guardado.


In [41]:
# Esta herramienta de descarga existe en Google Colab.
from google.colab import files

# Primer archivo: Pipeline ya entrenada (permite clasificar reseñas nuevas).
files.download("butaca_emocional_modelo_v2.joblib")
# Segundo archivo: métricas y términos del entrenamiento (permite dibujar gráficos).
files.download("metricas_v2.json")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 11. Conclusiones del experimento — resultados reales

El proyecto **💺 Butaca Emocional** utiliza reseñas de películas en español para aprender a diferenciar opiniones **positivas** y **negativas**. Partimos de **50.000 reseñas** y, después de la limpieza, utilizamos **49.599**. Entrenamos el modelo con **39.679** y reservamos **9.920** para examinarlo sin mostrarle antes esas respuestas.

El clasificador alcanzó una **exactitud del 87,64 %**: acertó **8.694 reseñas** y se equivocó en **1.226**. En concreto, identificó correctamente **4.294 negativas** y **4.400 positivas**; confundió **649 negativas con positivas** y **577 positivas con negativas**.

Entre los términos asociados a opiniones positivas destacan **«refrescante»**, **«perfecto»** y **«excelente»**. Entre los asociados a negativas aparecen **«decepción»**, **«no vale»** y **«horrible»**. Estos pesos son asociaciones aprendidas por el modelo, no reglas universales sobre el significado de cada frase.

**¿Qué hemos conseguido?** Hemos recorrido las fases de un proyecto de aprendizaje automático: cargar datos, limpiarlos, separar entrenamiento y prueba, transformar texto en números, entrenar, evaluar, interpretar y exportar los archivos necesarios para la aplicación Streamlit.

**Limitaciones:** el modelo solo distingue entre positivo y negativo; no predice una categoría neutra. Las ironías, negaciones y frases ambiguas pueden provocar errores. Los resultados de prueba describen este conjunto de reseñas, no garantizan el mismo rendimiento con todos los comentarios nuevos.

### 🗣️ Explicación sencilla para presentar el proyecto

> Hemos enseñado a un programa a distinguir reseñas positivas y negativas de películas. Primero le dimos ejemplos para aprender y luego le pusimos un examen con 9.920 reseñas nuevas. Acertó aproximadamente 88 de cada 100. Finalmente guardamos el modelo para que la aplicación Butaca Emocional pueda analizar opiniones escritas por los usuarios.